
# 🧭 Week 3 — Pipelines, Modularity & Debugging by Decomposition

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week3_pipelines_modularity.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

---

## 🎯 Learning Objectives

By the end of this lecture, you should be able to:

1. Describe why modular pipelines are more reliable than single long prompts.
2. Build a basic multi-step LLM system using `dspy`.
3. Explain how modularity enables debugging and measurement.
4. Identify points of failure within a multi-step system.
5. Extend your mental model of the **Unifying Diagram (v2)**.


In [1]:
# @title 🔧 Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append("/content/main")
print("✅ Environment ready!")


✅ Environment ready!



## 🔁 Lab 2 Recap — What Changed When Temperature Increased?

In Lab 2, you observed how **temperature** affects diversity.  
Today, we'll shift focus from *how the model samples* to *how we structure our systems*.

**Discussion Prompt:**  
> When temperature increased, what specific kinds of changes did you notice?  
> Tone? Creativity? Factual drift?



## 🧩 From Monolithic Prompts to Modular Pipelines

A *monolithic* prompt tries to do everything at once — it's hard to test, debug, or improve.

A *modular pipeline* breaks a task into smaller steps (e.g., "extract key ideas" → "simplify for students").  
Each step becomes a **testable unit** in our system.

Let's look at how `dspy` supports this kind of modular design.


In [3]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.2/285.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 71.4 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.


In [4]:
# 🧠 Example 1: A minimal dspy predictor
import dspy

# Define a simple question-answer predictor
predict = dspy.Predict("question -> answer")

# Run it
result = predict(question="What is the capital of France?")
print(result.answer)


ValueError: No LM is loaded. Please configure the LM using `dspy.configure(lm=dspy.LM(...))`. e.g, `dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))`

In [ ]:
# 🧩 Example 2: Adding structure — sentiment classification
sentiment_predict = dspy.Predict("sentence -> sentiment: bool")

positive = sentiment_predict(sentence="I love working with LLMs!")
negative = sentiment_predict(sentence="This code keeps crashing.")

print("Positive sentiment:", positive.sentiment)
print("Negative sentiment:", negative.sentiment)


In [ ]:
# 🧩 Example 3: Building a two-step pipeline

extract = dspy.Predict("paragraph -> key_points")
summarize = dspy.Predict("key_points -> simplified_summary")

paragraph = "Photosynthesis converts light energy into chemical energy in plants."

# Step 1: Extract key points
key_points = extract(paragraph=paragraph).key_points

# Step 2: Simplify for students
summary = summarize(key_points=key_points).simplified_summary

print("Summary:", summary)



## 🧠 Mini-Lecture — Debugging by Decomposition

When something fails, **which part failed**?  
If you have one big prompt, you can’t tell.

Modularity allows **failure localization** — knowing *which step* caused an issue.

| Concept | Example | Takeaway |
|----------|----------|-----------|
| Break tasks into steps | “Extract → Rewrite → Summarize” | Easier to test and fix |
| Observe intermediate outputs | Log or print results after each step | Transparency |
| Add small metrics per step | Length, keyword count, etc. | Quantitative checks |

> In AI Engineering, most bugs are system-level — not model-level.



## 🧩 Unifying Diagram v2 — Multi-Step Pipeline

```mermaid
graph TD
    U["User Input"] --> IH["Input Handling"]
    IH --> P1["Step 1: Extract Info"]
    P1 --> P2["Step 2: Summarize"]
    P2 --> LLM["Core LLM Model"]
    LLM --> OP["Output Processing"]
    OP --> M["Monitoring & Evaluation"]
```



## 💭 Reflection Prompts

1. What advantages did you notice when using typed predictions (like `sentiment: bool`)?  
2. How does `Predict("input -> output")` encourage modular thinking?  
3. Where could you insert evaluation or monitoring in this system?



<details>
<summary>🧑‍🏫 <b>Instructor Notes</b></summary>

- Encourage students to modify the `Predict` strings and rerun the cells.  
- Use live examples (like summarizing their own text) to make debugging concrete.  
- If dspy is unavailable in Colab, mock the results with simple print statements.  
- Reinforce that modularity precedes fine-tuning: structure first, train later.

</details>
